In [18]:
!pip install keras-tuner

In [19]:
# Step 1:-  Import Libraries
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout #to drop neurons
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelBinarizer
from sklearn.metrics import classification_report

In [20]:
# Step 2:- Load Dataset
data = pd.read_csv('/content/mobile_price_classification.csv')

In [21]:
#Step 3:- Split data and X and y
X = data.drop('price_range',axis=1)
X

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,primary_camera,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
0,842,0,2.2,0,1,0,7,0.6,188,2,2,20,756,2549,9,7,19,0,0,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,6,905,1988,2631,17,3,7,1,1,0
2,563,1,0.5,1,2,1,41,0.9,145,5,6,1263,1716,2603,11,2,9,1,1,0
3,615,1,2.5,0,0,0,10,0.8,131,6,9,1216,1786,2769,16,8,11,1,0,0
4,1821,1,1.2,0,13,1,44,0.6,141,2,14,1208,1212,1411,8,2,15,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,794,1,0.5,1,0,1,2,0.8,106,6,14,1222,1890,668,13,4,19,1,1,0
1996,1965,1,2.6,1,0,0,39,0.2,187,4,3,915,1965,2032,11,10,16,1,1,1
1997,1911,0,0.9,1,1,1,36,0.7,108,8,3,868,1632,3057,9,1,5,1,1,0
1998,1512,0,0.9,0,4,1,46,0.1,145,5,5,336,670,869,18,10,19,1,1,1


In [22]:
y = data['price_range']
y

,price_range
0,1
1,2
2,2
3,2
4,1
...,...
1995,0
1996,2
1997,3
1998,0


In [23]:
# Step 4:- One-hot encode labels using LabelBinarizer
lb = LabelBinarizer()
y = lb.fit_transform(y)
y

array([[0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 1, 0],
       ...,
       [0, 0, 0, 1],
       [1, 0, 0, 0],
       [0, 0, 0, 1]])

In [24]:
# Step 5:- Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape

(1600, 20)

In [25]:
# Step 6:- # 4. Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [26]:
# step 7:- Build ANN model
"""model = Sequential([
    Dense(64,input_shape=(X_train.shape[1],), activation='relu'),   # hidden layer 1
    Dense(32, activation='relu'), # hidden layer 2
    Dense(16, activation='relu'), # hidden layer 3
    Dense(4, activation='softmax')  # Output Layer ,Since we have multiple class we have used softmax
])"""

"model = Sequential([\n    Dense(64,input_shape=(X_train.shape[1],), activation='relu'),   # hidden layer 1\n    Dense(32, activation='relu'), # hidden layer 2\n    Dense(16, activation='relu'), # hidden layer 3\n    Dense(4, activation='softmax')  # Output Layer ,Since we have multiple class we have used softmax\n])"

In [29]:
def build_model(hp):
    model = Sequential()

    # Tune number of neurons
    model.add(Dense(
        units=hp.Int('units_1', min_value=8, max_value=64, step=8),
        activation='relu',
        input_shape=(20,)
    )) #hidden layer1

    # Tune second hidden layer neurons
    model.add(Dense(
        units=hp.Int('units_2', min_value=8, max_value=64, step=8),
        activation='relu'
    ))

    # Tune dropout
    model.add(Dropout(
        rate=hp.Choice('dropout_rate', values=[0.2, 0.3, 0.4])
    ))
    model.add(Dense(4, activation='softmax'))



    # Tune learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [30]:
# Initialize Tuner=1
tuner = kt.BayesianOptimization(
    build_model,
    objective='val_accuracy',
    max_trials=10,
)

# Search Best Hyperparameters
tuner.search(
    X_train, y_train,
    epochs=50,
    validation_split=0.2,
    batch_size=16,
    verbose=1
)

# Get Best Model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate on Test Set
loss, accuracy = best_model.evaluate(X_test, y_test)
print("\nBest Model Test Accuracy:", accuracy)

# Show Best Hyperparameters
best_hp = tuner.get_best_hyperparameters(1)[0]
print("\nBest Hyperparameters:")
print("Units Layer 1:", best_hp.get('units_1'))
print("Units Layer 2:", best_hp.get('units_2'))
print("Dropout Rate:", best_hp.get('dropout_rate'))
print("Learning Rate:", best_hp.get('learning_rate'))

# Classification Report
y_pred = np.argmax(best_model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

Trial 10 Complete [00h 00m 19s]
val_accuracy: 0.921875

Best val_accuracy So Far: 0.9437500238418579
Total elapsed time: 00h 06m 28s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9280 - loss: 0.1791  

Best Model Test Accuracy: 0.9325000047683716

Best Hyperparameters:
Units Layer 1: 40
Units Layer 2: 24
Dropout Rate: 0.2
Learning Rate: 0.001
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.92      0.95       105
           1       0.86      0.95      0.90        91
           2       0.94      0.88      0.91        92
           3       0.96      0.97      0.96       112

    accuracy                           0.93       400
   macro avg       0.93      0.93      0.93       400
weighted avg       0.93      0.93      0.93       400



In [31]:
# Rerun with updated value
#Load Dataset
data = pd.read_csv('/content/mobile_price_classification.csv')

In [32]:
#  Split data and X and y
X = data.drop('price_range',axis=1)
X

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,primary_camera,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
0,842,0,2.2,0,1,0,7,0.6,188,2,2,20,756,2549,9,7,19,0,0,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,6,905,1988,2631,17,3,7,1,1,0
2,563,1,0.5,1,2,1,41,0.9,145,5,6,1263,1716,2603,11,2,9,1,1,0
3,615,1,2.5,0,0,0,10,0.8,131,6,9,1216,1786,2769,16,8,11,1,0,0
4,1821,1,1.2,0,13,1,44,0.6,141,2,14,1208,1212,1411,8,2,15,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,794,1,0.5,1,0,1,2,0.8,106,6,14,1222,1890,668,13,4,19,1,1,0
1996,1965,1,2.6,1,0,0,39,0.2,187,4,3,915,1965,2032,11,10,16,1,1,1
1997,1911,0,0.9,1,1,1,36,0.7,108,8,3,868,1632,3057,9,1,5,1,1,0
1998,1512,0,0.9,0,4,1,46,0.1,145,5,5,336,670,869,18,10,19,1,1,1


In [33]:
y = data['price_range']
y

,price_range
0,1
1,2
2,2
3,2
4,1
...,...
1995,0
1996,2
1997,3
1998,0


In [34]:
# One-hot encode labels using LabelBinarizer
lb = LabelBinarizer()
y = lb.fit_transform(y)
y

array([[0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 1, 0],
       ...,
       [0, 0, 0, 1],
       [1, 0, 0, 0],
       [0, 0, 0, 1]])

In [35]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape

(1600, 20)

In [36]:
X_train.shape[0]
X_train.shape[1]

20

In [37]:
# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [38]:
# Build ANN model
model = Sequential([
    Dense(40,input_shape=(X_train.shape[1],), activation='relu'),   # hidden layer 1
    Dense(24, activation='relu'), # hidden layer 2
    Dense(4, activation='softmax')  # Output Layer ,Since we have multiple class we have used softmax
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
#Compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [40]:
#Train model
history = model.fit(X_train, y_train, epochs=35, batch_size=5, validation_data=(X_test, y_test), verbose=1)

Epoch 1/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.3278 - loss: 1.3200 - val_accuracy: 0.6700 - val_loss: 0.7895
Epoch 2/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7328 - loss: 0.6960 - val_accuracy: 0.8250 - val_loss: 0.4503
Epoch 3/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8660 - loss: 0.3974 - val_accuracy: 0.8875 - val_loss: 0.3192
Epoch 4/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9169 - loss: 0.2737 - val_accuracy: 0.9025 - val_loss: 0.2584
Epoch 5/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9276 - loss: 0.2100 - val_accuracy: 0.9050 - val_loss: 0.2363
Epoch 6/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9619 - loss: 0.1564 - val_accuracy: 0.8950 - val_loss: 0.2265
Epoch 7/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9630 - loss: 0.1325 - val_accuracy: 0.9200 - val_loss: 0.1906
Epoch 8/35
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9616 - loss: 0.1212 - val_accuracy: 0.

In [41]:
#Evaluate model
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.2f}")

Test Accuracy: 0.93
